In [1]:
!pip install -q pillow numpy scikit-image torch torchvision matplotlib tqdm


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\qweqw\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import random
import time
import zipfile

import numpy as np
from PIL import Image, ImageFilter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


# фиксируем случайность
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

# Путь к данным.
# Локально: твой путь.
# В Colab: например "/content/data" или путь к Google Drive.
DATA_DIR = r"D:\prog\python_p\ii\2p_2"
# DATA_DIR = "/content/data"

GRID = 24
FS = 20
IMG_SIZE = GRID * FS  # 480

TRAIN_INPUT_DIR = os.path.join(DATA_DIR, "train", "inputs")
TRAIN_TARGET_DIR = os.path.join(DATA_DIR, "train", "targets")
TEST_DIR = os.path.join(DATA_DIR, "test")

SUBMISSION_DIR = os.path.join(DATA_DIR, "submission_final")
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# главные настройки
CONFIG = {
    # сколько картинок оставить для проверки
    "val_size": 700,

    # пазл
    "puzzle_max_images": 200,
    "puzzle_epochs": 3,
    "puzzle_batch_size": 128,
    "puzzle_lr": 1e-3,
    "puzzle_num_starts": 20,

    # реставрация
    "restore_max_images": 200,
    "restore_epochs": 5,
    "restore_batch_size": 4,
    "restore_lr": 1e-3,

    # сколько валидационных картинок проверять
    "eval_images": 3,

    # сколько стартов пробить на тесте
    "submission_num_starts": 10,
}

# Если False - сделает только 5 тестовых картинок для проверки.
# Для финальной сдачи нужно поставить True.
RUN_FULL_SUBMISSION = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


C:\Users\qweqw\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_image(path):
    # открываем картинку и переводим в RGB
    return np.array(Image.open(path).convert("RGB"))


def get_png_names(directory):
    # список всех png в папке
    return sorted([
        name for name in os.listdir(directory)
        if name.lower().endswith(".png")
    ])


def extract_fragments(img):
    # режем картинку 480x480 на 576 кусков 20x20
    fragments = []

    for r in range(GRID):
        for c in range(GRID):
            frag = img[r * FS:(r + 1) * FS, c * FS:(c + 1) * FS]
            fragments.append(frag)

    return np.array(fragments, dtype=np.uint8)


def make_pair_canvas(frag_a, frag_b, orientation="right"):
    # делаем картинку 40x40 из двух кусков
    canvas = np.zeros((40, 40, 3), dtype=np.uint8)

    if orientation == "right":
        canvas[0:20, 0:20] = frag_a
        canvas[0:20, 20:40] = frag_b
    else:
        canvas[0:20, 0:20] = frag_a
        canvas[20:40, 0:20] = frag_b

    return canvas


def grid_to_canvas(fragments, grid):
    # собираем картинку 480x480 из сетки кусков
    canvas = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

    for r in range(GRID):
        for c in range(GRID):
            frag_id = grid[r, c]
            canvas[r * FS:(r + 1) * FS, c * FS:(c + 1) * FS] = fragments[frag_id]

    return canvas


def calc_ssim(img1, img2):
    # считаем SSIM между двумя картинками
    return ssim(img1, img2, channel_axis=2, data_range=255)

In [4]:
input_names = get_png_names(TRAIN_INPUT_DIR)
target_names = get_png_names(TRAIN_TARGET_DIR)
test_names = get_png_names(TEST_DIR)

print("Train inputs:", len(input_names))
print("Train targets:", len(target_names))
print("Test images:", len(test_names))

# проверяем, что входы и ответы совпадают
assert len(input_names) == len(target_names), "Разное число файлов в inputs и targets"
assert set(input_names) == set(target_names), "Имена файлов в inputs и targets не совпадают"

# мешаем и делим данные
all_names = input_names.copy()
random.shuffle(all_names)

val_size = CONFIG["val_size"]

val_names = all_names[:val_size]
train_names = all_names[val_size:]

print("Train split:", len(train_names))
print("Val split:", len(val_names))

Train inputs: 7000
Train targets: 7000
Test images: 700
Train split: 6300
Val split: 700


In [5]:
def corrupt_fragment(frag):
    # портим один кусок: яркость, контраст, шум, блюр

    x = frag.astype(np.float32)

    # контраст
    contrast = np.random.uniform(0.7, 1.3)
    x = (x - 128.0) * contrast + 128.0

    # яркость
    x += np.random.uniform(-30.0, 30.0)

    x = np.clip(x, 0, 255).astype(np.uint8)

    # легкое размытие
    if np.random.random() < 0.7:
        x = np.array(
            Image.fromarray(x).filter(
                ImageFilter.GaussianBlur(radius=1)
            )
        )

    # шум
    sigma = np.random.uniform(15.0, 45.0)
    noise = np.random.normal(0.0, sigma, x.shape).astype(np.float32)

    x = x.astype(np.float32) + noise
    x = np.clip(x, 0, 255).astype(np.uint8)

    return x


def make_dirty_assembled(clean_img):
    # берем чистую картинку, портим каждый кусок, собираем обратно
    frags = extract_fragments(clean_img)
    dirty = np.zeros_like(clean_img)

    idx = 0
    for r in range(GRID):
        for c in range(GRID):
            dirty[r * FS:(r + 1) * FS, c * FS:(c + 1) * FS] = corrupt_fragment(frags[idx])
            idx += 1

    return dirty

In [6]:
class PairCNN(nn.Module):
    # маленькая сеть для пары кусков
    # 0 - не соседи
    # 1 - второй справа
    # 2 - второй снизу

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 3),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


class PuzzleDataset(Dataset):
    # датасет пар кусков
    # берем чистые картинки и сами портим куски на лету

    def __init__(self, names, target_dir, max_images=200):
        self.frags_list = []
        self.items = []

        for name in names[:max_images]:
            img = load_image(os.path.join(target_dir, name))
            frags = extract_fragments(img)

            img_id = len(self.frags_list)
            self.frags_list.append(frags)

            positive_pairs = []

            # настоящие соседи
            for r in range(GRID):
                for c in range(GRID):
                    idx = r * GRID + c

                    if c + 1 < GRID:
                        positive_pairs.append((idx, idx + 1, 1, "right"))

                    if r + 1 < GRID:
                        positive_pairs.append((idx, idx + GRID, 2, "below"))

            # случайные плохие пары
            negative_pairs = []
            for _ in range(len(positive_pairs)):
                i1, i2 = random.sample(range(GRID * GRID), 2)
                orientation = random.choice(["right", "below"])
                negative_pairs.append((i1, i2, 0, orientation))

            for i, j, label, orientation in positive_pairs:
                self.items.append((img_id, i, j, label, orientation))

            for i, j, label, orientation in negative_pairs:
                self.items.append((img_id, i, j, label, orientation))

        random.shuffle(self.items)
        print(f"Пар для пазла: {len(self.items):,}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_id, i, j, label, orientation = self.items[idx]

        frags = self.frags_list[img_id]

        a = corrupt_fragment(frags[i])
        b = corrupt_fragment(frags[j])

        pair = make_pair_canvas(a, b, orientation=orientation)

        x = torch.from_numpy(pair.copy()).permute(2, 0, 1).float() / 255.0
        y = int(label)

        return x, y


def train_puzzle(model, dataset, epochs, batch_size, lr):
    # обучение модели пазла

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()

        total_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(loader, desc=f"Puzzle epoch {epoch + 1}/{epochs}")

        for x, y in pbar:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += x.size(0)

            pbar.set_postfix(
                loss=total_loss / total,
                acc=correct / total
            )

    return model

In [7]:
@torch.no_grad()
def compute_pair_scores(model, fragments, batch_size=2048):
    # считаем вероятности для всех пар кусков

    model.eval()
    n = len(fragments)

    right_scores = np.zeros((n, n), dtype=np.float32)
    below_scores = np.zeros((n, n), dtype=np.float32)

    for i in range(n):
        others = [j for j in range(n) if j != i]

        pairs_right = np.stack([
            make_pair_canvas(fragments[i], fragments[j], orientation="right")
            for j in others
        ])

        pairs_below = np.stack([
            make_pair_canvas(fragments[i], fragments[j], orientation="below")
            for j in others
        ])

        # право
        x_right = torch.from_numpy(pairs_right).permute(0, 3, 1, 2).float() / 255.0
        probs_right = []

        for start in range(0, len(x_right), batch_size):
            batch = x_right[start:start + batch_size].to(device)
            logits = model(batch)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            probs_right.append(probs)

        probs_right = np.concatenate(probs_right, axis=0)
        right_scores[i, others] = probs_right[:, 1]

        # низ
        x_below = torch.from_numpy(pairs_below).permute(0, 3, 1, 2).float() / 255.0
        probs_below = []

        for start in range(0, len(x_below), batch_size):
            batch = x_below[start:start + batch_size].to(device)
            logits = model(batch)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            probs_below.append(probs)

        probs_below = np.concatenate(probs_below, axis=0)
        below_scores[i, others] = probs_below[:, 2]

    return right_scores, below_scores


def choose_start_candidates(right_scores, below_scores, num_starts=20):
    # ищем куски, похожие на левый верхний угол

    outgoing = right_scores.max(axis=1) + below_scores.max(axis=1)
    incoming = right_scores.max(axis=0) + below_scores.max(axis=0)

    corner_score = outgoing - incoming
    candidates = np.argsort(-corner_score)

    return candidates[:num_starts]


def assemble_with_start_fast(right_scores, below_scores, start_fragment):
    # жадная сборка сетки от стартового куска

    n = right_scores.shape[0]

    grid = -np.ones((GRID, GRID), dtype=int)
    used = np.zeros(n, dtype=bool)

    grid[0, 0] = start_fragment
    used[start_fragment] = True

    for r in range(GRID):
        for c in range(GRID):
            if r == 0 and c == 0:
                continue

            scores = np.zeros(n, dtype=np.float32)
            count = 0

            # смотрим на левого соседа
            if c > 0 and grid[r, c - 1] >= 0:
                left_id = grid[r, c - 1]
                scores += right_scores[left_id]
                count += 1

            # смотрим на верхнего соседа
            if r > 0 and grid[r - 1, c] >= 0:
                top_id = grid[r - 1, c]
                scores += below_scores[top_id]
                count += 1

            if count > 0:
                scores /= count

            scores[used] = -1e18

            best_idx = int(np.argmax(scores))
            grid[r, c] = best_idx
            used[best_idx] = True

    # оценка собранной сетки
    total_score = 0.0
    total_count = 0

    for r in range(GRID):
        for c in range(GRID):
            cur = grid[r, c]

            if c + 1 < GRID:
                right_id = grid[r, c + 1]
                total_score += right_scores[cur, right_id]
                total_count += 1

            if r + 1 < GRID:
                below_id = grid[r + 1, c]
                total_score += below_scores[cur, below_id]
                total_count += 1

    avg_score = total_score / max(1, total_count)

    return grid, avg_score


def solve_puzzle(model, image, num_starts=20):
    # полная сборка одной картинки

    fragments = extract_fragments(image)

    right_scores, below_scores = compute_pair_scores(
        model=model,
        fragments=fragments,
        batch_size=2048
    )

    candidates = choose_start_candidates(
        right_scores=right_scores,
        below_scores=below_scores,
        num_starts=num_starts
    )

    best_grid = None
    best_score = -1e18

    for start_id in candidates:
        grid, score = assemble_with_start_fast(
            right_scores=right_scores,
            below_scores=below_scores,
            start_fragment=int(start_id)
        )

        if score > best_score:
            best_score = score
            best_grid = grid

    return grid_to_canvas(fragments, best_grid)

In [8]:
class ConvBlock(nn.Module):
    # простой блок: две свертки
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),

            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.block(x)


class SmallUNet(nn.Module):
    # небольшая сеть для очистки картинки
    # учит шум и швы между кусками

    def __init__(self):
        super().__init__()

        self.enc1 = ConvBlock(3, 32)
        self.enc2 = ConvBlock(32, 64)
        self.enc3 = ConvBlock(64, 128)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(128, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(64, 32)

        self.out = nn.Conv2d(32, 3, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        out = self.out(d1)

        # residual: возвращаем исходную картинку плюс найденная поправка
        return torch.clamp(x + out, 0.0, 1.0)


class RestorationDataset(Dataset):
    # датасет для реставрации
    # вход: чистая картинка, испорченная по кускам
    # выход: исходная чистая картинка

    def __init__(self, names, target_dir, max_images=200):
        self.names = names[:max_images]
        self.target_dir = target_dir

        print(f"Картинок для реставрации: {len(self.names)}")

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        clean = load_image(os.path.join(self.target_dir, name))
        dirty = make_dirty_assembled(clean)

        x = torch.from_numpy(dirty.copy()).permute(2, 0, 1).float() / 255.0
        y = torch.from_numpy(clean.copy()).permute(2, 0, 1).float() / 255.0

        return x, y


def train_restorer(model, dataset, epochs, batch_size, lr):
    # обучение реставратора

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    for epoch in range(epochs):
        model.train()

        total_loss = 0.0
        total = 0

        pbar = tqdm(loader, desc=f"Restorer epoch {epoch + 1}/{epochs}")

        for x, y in pbar:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            pred = model(x)
            loss = criterion(pred, y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)
            total += x.size(0)

            pbar.set_postfix(loss=total_loss / total)

    return model


def restore_image(model, img):
    # чистим одну картинку

    model.eval()

    x = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
    x = x.unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(x)

    out = pred.squeeze(0).permute(1, 2, 0).cpu().numpy()
    out = np.clip(out * 255.0, 0, 255).astype(np.uint8)

    return out

In [9]:
print("Создаю датасет пазла")
puzzle_dataset = PuzzleDataset(
    names=train_names,
    target_dir=TRAIN_TARGET_DIR,
    max_images=CONFIG["puzzle_max_images"]
)

print("Создаю модель пазла")
puzzle_model = PairCNN().to(device)

print("Параметров модели пазла:", sum(p.numel() for p in puzzle_model.parameters()))

puzzle_model = train_puzzle(
    model=puzzle_model,
    dataset=puzzle_dataset,
    epochs=CONFIG["puzzle_epochs"],
    batch_size=CONFIG["puzzle_batch_size"],
    lr=CONFIG["puzzle_lr"]
)

torch.cuda.empty_cache()

Создаю датасет пазла
Пар для пазла: 441,600
Создаю модель пазла
Параметров модели пазла: 102147


Puzzle epoch 3/3: 100%|██████████| 3450/3450 [01:33<00:00, 37.04it/s, acc=0.845, loss=0.355]


In [10]:
print("Создаю датасет реставрации")
restore_dataset = RestorationDataset(
    names=train_names,
    target_dir=TRAIN_TARGET_DIR,
    max_images=CONFIG["restore_max_images"]
)

print("Создаю модель реставрации")
restorer = SmallUNet().to(device)

print("Параметров модели реставрации:", sum(p.numel() for p in restorer.parameters()))

restorer = train_restorer(
    model=restorer,
    dataset=restore_dataset,
    epochs=CONFIG["restore_epochs"],
    batch_size=CONFIG["restore_batch_size"],
    lr=CONFIG["restore_lr"]
)

torch.cuda.empty_cache()

Создаю датасет реставрации
Картинок для реставрации: 200
Создаю модель реставрации
Параметров модели реставрации: 1928483


Restorer epoch 5/5: 100%|██████████| 50/50 [00:16<00:00,  3.11it/s, loss=0.0684]


In [11]:
EXPERIMENTS = []

def evaluate_full_pipeline(puzzle_model, restorer, names, num_images=3, num_starts=20):
    # проверяем: сборка пазла + реставрация

    scores = []

    for name in names[:num_images]:
        input_path = os.path.join(TRAIN_INPUT_DIR, name)
        target_path = os.path.join(TRAIN_TARGET_DIR, name)

        input_img = load_image(input_path)
        target_img = load_image(target_path)

        print("Обрабатываю:", name)

        assembled = solve_puzzle(
            model=puzzle_model,
            image=input_img,
            num_starts=num_starts
        )

        restored = restore_image(
            model=restorer,
            img=assembled
        )

        score = calc_ssim(target_img, restored)
        scores.append(score)

        print("SSIM:", round(score, 4))

    mean_score = float(np.mean(scores)) if scores else 0.0
    print("Mean SSIM:", round(mean_score, 4))

    return mean_score


mean_val_ssim = evaluate_full_pipeline(
    puzzle_model=puzzle_model,
    restorer=restorer,
    names=val_names,
    num_images=CONFIG["eval_images"],
    num_starts=CONFIG["puzzle_num_starts"]
)

# сохраняем эксперимент в журнал
EXPERIMENTS.append({
    "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    "puzzle_max_images": CONFIG["puzzle_max_images"],
    "puzzle_epochs": CONFIG["puzzle_epochs"],
    "restore_max_images": CONFIG["restore_max_images"],
    "restore_epochs": CONFIG["restore_epochs"],
    "val_mean_ssim": mean_val_ssim,
})

print()
print("Журнал экспериментов:")
for exp in EXPERIMENTS:
    print(exp)

Обрабатываю: img_000434.png
SSIM: 0.1911
Обрабатываю: img_002772.png
SSIM: 0.2076
Обрабатываю: img_003401.png
SSIM: 0.2834
Mean SSIM: 0.2274

Журнал экспериментов:
{'time': '2026-09-16 17:18:46', 'puzzle_max_images': 200, 'puzzle_epochs': 3, 'restore_max_images': 200, 'restore_epochs': 5, 'val_mean_ssim': 0.22737013685732876}


In [12]:
puzzle_model_path = os.path.join(DATA_DIR, "puzzle_model.pth")
restorer_model_path = os.path.join(DATA_DIR, "restorer_model.pth")

torch.save(puzzle_model.state_dict(), puzzle_model_path)
torch.save(restorer.state_dict(), restorer_model_path)

print("Сохранено:")
print(puzzle_model_path)
print(restorer_model_path)

Сохранено:
D:\prog\python_p\ii\2p_2\puzzle_model.pth
D:\prog\python_p\ii\2p_2\restorer_model.pth


In [13]:
def clear_folder(folder):
    # удаляем старые файлы из папки сабмита
    if not os.path.exists(folder):
        os.makedirs(folder)
        return

    for filename in os.listdir(folder):
        file_path = os.path.join(folder, filename)
        if os.path.isfile(file_path):
            os.remove(file_path)


def make_submission(puzzle_model, restorer, test_names, out_dir, zip_path, full=False):
    clear_folder(out_dir)

    if full:
        names = test_names
    else:
        names = test_names[:5]

    print(f"Сделаю сабмит для {len(names)} изображений")

    for name in tqdm(names):
        test_path = os.path.join(TEST_DIR, name)
        img = load_image(test_path)

        assembled = solve_puzzle(
            model=puzzle_model,
            image=img,
            num_starts=CONFIG["submission_num_starts"]
        )

        restored = restore_image(
            model=restorer,
            img=assembled
        )

        save_path = os.path.join(out_dir, name)
        Image.fromarray(restored).save(save_path)

    # пакуем в zip
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for filename in sorted(os.listdir(out_dir)):
            file_path = os.path.join(out_dir, filename)
            zf.write(file_path, arcname=filename)

    print("Готовый архив:")
    print(zip_path)


zip_path = os.path.join(DATA_DIR, "submission_final.zip")

make_submission(
    puzzle_model=puzzle_model,
    restorer=restorer,
    test_names=test_names,
    out_dir=SUBMISSION_DIR,
    zip_path=zip_path,
    full=RUN_FULL_SUBMISSION
)

# проверка числа файлов
saved_files = sorted(os.listdir(SUBMISSION_DIR))
print("Файлов в сабмите:", len(saved_files))

if RUN_FULL_SUBMISSION:
    assert len(saved_files) == len(test_names), "В сабмите не все файлы"

Сделаю сабмит для 700 изображений


100%|██████████| 700/700 [2:28:44<00:00, 12.75s/it]  


Готовый архив:
D:\prog\python_p\ii\2p_2\submission_final.zip
Файлов в сабмите: 700
